In [ ]:
#| hide
from cassava.data import *

# cassava

> Predict δ13C in cassava leaves from mid-infrared spectra.

**Work in progress.** The analysis isn't finished. The findings below come from a first round of models, and the growing conditions of the batches are still missing. Expect the results and conclusions to change.

This project tests whether mid-infrared (MIR) spectra of cassava leaves can predict their carbon isotope ratio, δ13C. The data are absorbance spectra and δ13C lab values for 1158 leaf samples from seven batches. Each batch is a set of plants grown under its own conditions and scanned at its own time, on the same spectrometer. The analysis is a set of [nbdev](https://nbdev.fast.ai) notebooks, and the package `cassava` holds the data loaders.

## Why δ13C

Cassava is a C3 plant. During photosynthesis, its carbon-fixing enzyme, Rubisco, fixes CO2 that contains the light isotope 12C more readily than CO2 that contains 13C. How strongly the leaf favours 12C depends on the CO2 concentration inside the leaf, compared with the air outside.

A plant short of water closes its stomata to save water. The CO2 concentration inside the leaf then falls, and the leaf favours 12C less. Its δ13C rises, which means it becomes less negative.

Leaf δ13C therefore records the balance between carbon gain and water loss while the leaf grew. Plant scientists use it as a measure of water-use efficiency and of a plant's response to water stress. In wheat, for example, breeders have used it to select varieties that use water more efficiently.

Measuring δ13C needs isotope-ratio mass spectrometry, which is slow and costly for each sample. MIR spectra are fast and cheap to collect. A model that predicts δ13C from spectra would let breeders and physiologists screen many more plants.

## Findings so far

Batch alone explains 68% of the δ13C variance. A PLS model on the spectra reaches an R² of 0.77 on a random split stratified by batch. Most of that score comes from recognising the batch.

The model can't predict the δ13C level of a batch it hasn't seen. Under leave-one-batch-out validation, its R² is -0.01.

Inside batches, the spectra carry δ13C information. A local PLS model, calibrated on samples from one batch, explains 36% to 73% of the δ13C variation in six of the seven batches.

MIRS can rank samples within a trial when some samples from that trial have lab values. It can't replace lab measurements for a new trial.

The [report](https://franckalbinet.github.io/cassava/report.html) gives the full results, the validation design and the open questions.

## Install

Clone the repository and create its environment with [uv](https://docs.astral.sh/uv/):

```sh
git clone https://github.com/franckalbinet/cassava.git
cd cassava
uv sync
```

## Data

The data files are not in the repository. Put them in a `data/` folder at the project root:

```
data/specs_trim.csv
data/all_batches_wet_chem.csv
```

`specs_trim.csv` holds the absorbance spectra, with one row per wavenumber and one column per sample. `all_batches_wet_chem.csv` holds the δ13C value of each sample.

## Usage

`load_data` reads both files and keeps the samples present in both. It returns numpy arrays, ready for `soilspectfm` transforms and scikit-learn estimators:

In [ ]:
#| eval: false
d = load_data()
d.X.shape, d.ws[[0, -1]], d.y[:3], d.batch[:3]

((1158, 1711),
 array([ 651.8218, 3949.5   ]),
 array([-22.96865279, -23.89083784, -22.50169575]),
 array(['1456', '1456', '1456'], dtype='<U4'))

`d.X` holds one spectrum per row, `d.ws` the wavenumbers in cm⁻¹, `d.y` the δ13C values in ‰, and `d.batch` the batch of each sample.

## Notebooks

- [Data](https://franckalbinet.github.io/cassava/data.html) (`00_data`): loading and aligning the two files.
- [Exploratory data analysis](https://franckalbinet.github.io/cassava/eda.html) (`01_eda`): δ13C by batch, PCA of the spectra and the water vapour index.
- [PLS baseline](https://franckalbinet.github.io/cassava/pls.html) (`02_pls`): global, local and pooled PLS models under random and leave-one-batch-out validation.
- [Report](https://franckalbinet.github.io/cassava/report.html) (`03_report`): the findings.

## Development

Edit the notebooks in `nbs/`, not the generated `.py` files. Then run:

```sh
uv run nbdev-export    # regenerate the cassava package
uv run nbdev-test      # run the notebooks as tests
uv run nbdev-preview   # preview the documentation site
```

Cells that read the data have `#| eval: false`, because the data is not in the repository. `nbdev-test` skips them.